# 05 — Weather Data Preparation

## Dengue Disease Prediction for Sri Lanka

### Purpose

This notebook prepares the weekly weather dataset for monthly dengue forecasting.

The objectives are to:

- validate the standardized weekly weather dataset
- verify temporal and geographic coverage
- identify invalid, missing, or sentinel weather values
- examine reporting periods that cross calendar-month boundaries
- define variable-specific monthly aggregation rules
- convert weekly weather observations into monthly environmental features
- align the weather data with the 25 administrative districts used by the dengue forecasting dataset

Different weather variables require different aggregation strategies because averages, totals, minima and maxima have different statistical meanings.

No raw weather files are modified.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
print("Project root:", PROJECT_ROOT)
print("Interim directory:", INTERIM_DIR)

Project root: d:\Dengue_Project
Interim directory: d:\Dengue_Project\data\interim


In [3]:
weather_file = (
    INTERIM_DIR /
    "weather_weekly_geography_standardized.csv"
)
print("Weather file:", weather_file)
print("File exists:", weather_file.exists())

Weather file: d:\Dengue_Project\data\interim\weather_weekly_geography_standardized.csv
File exists: True


In [4]:
weather = pd.read_csv(weather_file)
print("Weather shape:", weather.shape)
weather.head()

Weather shape: (20332, 22)


,year,week,district,latitude,longitude,period_start,period_end,temperature_mean_c,temperature_min_c,temperature_max_c,...,humidity_max_percent,rainfall_total_mm,rainfall_max_hourly_mm,rainy_days,wind_speed_mean_kmh,wind_speed_max_kmh,source_file,source_area,reporting_area,administrative_district
0,2010,2,Ampara,7.29,81.67,2010-01-02,2010-01-08,25.06,21.9,28.6,...,91,5.8,0.6,4,12.47,21.1,Ampara_weekly_weather_2010_2024.csv,Ampara,Ampara,Ampara
1,2010,2,Anuradhapura,8.31,80.40,2010-01-02,2010-01-08,23.99,18.9,28.1,...,98,34.9,4.4,3,9.45,18.4,Anuradhapura_weekly_weather_2010_2024.csv,Anuradhapura,Anuradhapura,Anuradhapura
2,2010,2,Badulla,6.99,81.06,2010-01-02,2010-01-08,22.27,18.5,26.5,...,98,9.9,1.6,3,5.30,10.5,Badulla_weekly_weather_2010_2024.csv,Badulla,Badulla,Badulla
3,2010,2,Batticaloa,7.72,81.70,2010-01-02,2010-01-08,25.20,21.9,28.2,...,96,9.2,0.8,5,11.12,20.0,Batticaloa_weekly_weather_2010_2024.csv,Batticaloa,Batticaloa,Batticaloa
4,2010,2,Colombo,6.93,79.86,2010-01-02,2010-01-08,25.50,21.3,30.1,...,94,1.1,0.2,2,12.82,19.8,Colombo_weekly_weather_2010_2024.csv,Colombo,Colombo,Colombo


In [5]:
print("Weather columns:")
for column in weather.columns:
    print(column)

Weather columns:
year
week
district
latitude
longitude
period_start
period_end
temperature_mean_c
temperature_min_c
temperature_max_c
humidity_mean_percent
humidity_min_percent
humidity_max_percent
rainfall_total_mm
rainfall_max_hourly_mm
rainy_days
wind_speed_mean_kmh
wind_speed_max_kmh
source_file
source_area
reporting_area
administrative_district


In [6]:
required_columns = [
    "year",
    "week",
    "period_start",
    "period_end",
    "reporting_area",
    "administrative_district",
    "temperature_mean_c",
    "temperature_min_c",
    "temperature_max_c",
    "humidity_mean_percent",
    "humidity_min_percent",
    "humidity_max_percent",
    "rainfall_total_mm",
    "rainfall_max_hourly_mm",
    "rainy_days",
    "wind_speed_mean_kmh",
    "wind_speed_max_kmh"
]
missing_columns = [
    column
    for column in required_columns
    if column not in weather.columns
]

print("Missing required columns:", missing_columns)

Missing required columns: []


In [7]:
weather[required_columns].dtypes

year                         int64
week                         int64
period_start                   str
period_end                     str
reporting_area                 str
administrative_district        str
temperature_mean_c         float64
temperature_min_c          float64
temperature_max_c          float64
humidity_mean_percent      float64
humidity_min_percent         int64
humidity_max_percent         int64
rainfall_total_mm          float64
rainfall_max_hourly_mm     float64
rainy_days                   int64
wind_speed_mean_kmh        float64
wind_speed_max_kmh         float64
dtype: object

In [9]:
weather["start_date"] = pd.to_datetime(
    weather["period_start"],
    errors="coerce"
)
weather["end_date"] = pd.to_datetime(
    weather["period_end"],
    errors="coerce"
)
print(
    "Invalid start dates:",
    weather["start_date"].isna().sum()
)
print(
    "Invalid end dates:",
    weather["end_date"].isna().sum()
)

Invalid start dates: 0
Invalid end dates: 0


In [11]:
weather["reporting_days"] = (
    weather["end_date"] -
    weather["start_date"]
).dt.days + 1
weather["reporting_days"].value_counts().sort_index()
print(
    "Minimum reporting days:",
    weather["reporting_days"].min()
)
print(
    "Maximum reporting days:",
    weather["reporting_days"].max()
)

Minimum reporting days: 7
Maximum reporting days: 7


## 1. Weather Data Validation

The standardized weekly weather dataset is validated before temporal aggregation. Standard missing values, sentinel values, physically implausible values, temporal coverage and geographic completeness are examined separately.

In [12]:
weather_missing = (
    weather[required_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)
weather_missing

year                       0
week                       0
period_start               0
period_end                 0
reporting_area             0
administrative_district    0
temperature_mean_c         0
temperature_min_c          0
temperature_max_c          0
humidity_mean_percent      0
humidity_min_percent       0
humidity_max_percent       0
rainfall_total_mm          0
rainfall_max_hourly_mm     0
rainy_days                 0
wind_speed_mean_kmh        0
wind_speed_max_kmh         0
dtype: int64

In [13]:
weather_variables = [
    "temperature_mean_c",
    "temperature_min_c",
    "temperature_max_c",
    "humidity_mean_percent",
    "humidity_min_percent",
    "humidity_max_percent",
    "rainfall_total_mm",
    "rainfall_max_hourly_mm",
    "rainy_days",
    "wind_speed_mean_kmh",
    "wind_speed_max_kmh"
]

In [14]:
weather[weather_variables].dtypes

temperature_mean_c        float64
temperature_min_c         float64
temperature_max_c         float64
humidity_mean_percent     float64
humidity_min_percent        int64
humidity_max_percent        int64
rainfall_total_mm         float64
rainfall_max_hourly_mm    float64
rainy_days                  int64
wind_speed_mean_kmh       float64
wind_speed_max_kmh        float64
dtype: object

Search for common sentinel values

In [16]:
sentinel_values = [
    -9999,
    -999,
    -99,
    999,
    9999
]
sentinel_results = []
for column in weather_variables:
    for value in sentinel_values:
        count = (
            weather[column] == value
        ).sum()
        if count > 0:
            sentinel_results.append({
                "variable": column,
                "sentinel_value": value,
                "count": count
            })
sentinel_results = pd.DataFrame(
    sentinel_results
)
sentinel_results

""


Check basic physical consistency


In [17]:
temperature_order_errors = (
    (
        weather["temperature_min_c"] >
        weather["temperature_mean_c"]
    )
    |
    (
        weather["temperature_mean_c"] >
        weather["temperature_max_c"]
    )
).sum()
print(
    "Temperature ordering errors:",
    temperature_order_errors
)

Temperature ordering errors: 0


Humidity consistency

In [18]:
humidity_order_errors = (
    (
        weather["humidity_min_percent"] >
        weather["humidity_mean_percent"]
    )
    |
    (
        weather["humidity_mean_percent"] >
        weather["humidity_max_percent"]
    )
).sum()

print(
    "Humidity ordering errors:",
    humidity_order_errors
)

Humidity ordering errors: 0


In [19]:
humidity_out_of_range = (
    (weather["humidity_min_percent"] < 0) |
    (weather["humidity_max_percent"] > 100)
).sum()
print(
    "Humidity values outside 0–100%:",
    humidity_out_of_range
)

Humidity values outside 0–100%: 0


Check impossible negative quantities

In [20]:
negative_rainfall_total = (
    weather["rainfall_total_mm"] < 0
).sum()

negative_rainfall_max = (
    weather["rainfall_max_hourly_mm"] < 0
).sum()

negative_rainy_days = (
    weather["rainy_days"] < 0
).sum()

negative_wind_mean = (
    weather["wind_speed_mean_kmh"] < 0
).sum()

negative_wind_max = (
    weather["wind_speed_max_kmh"] < 0
).sum()
print(
    "Negative rainfall totals:",
    negative_rainfall_total
)
print(
    "Negative maximum hourly rainfall:",
    negative_rainfall_max
)
print(
    "Negative rainy days:",
    negative_rainy_days
)
print(
    "Negative mean wind speed:",
    negative_wind_mean
)
print(
    "Negative maximum wind speed:",
    negative_wind_max
)

Negative rainfall totals: 0
Negative maximum hourly rainfall: 0
Negative rainy days: 0
Negative mean wind speed: 0
Negative maximum wind speed: 0


Check rainy day consistency

In [21]:
rainy_day_range_errors = (
    (weather["rainy_days"] < 0) |
    (
        weather["rainy_days"] >
        weather["reporting_days"]
    )
).sum()
print(
    "Rainy-day range errors:",
    rainy_day_range_errors
)

Rainy-day range errors: 0


Check wind-speed ordering

In [22]:
wind_order_errors = (
    weather["wind_speed_mean_kmh"] >
    weather["wind_speed_max_kmh"]
).sum()
print(
    "Wind-speed ordering errors:",
    wind_order_errors
)

Wind-speed ordering errors: 0


Check rainfall ordering

In [23]:
rainfall_consistency_errors = (
    weather["rainfall_max_hourly_mm"] >
    weather["rainfall_total_mm"]
).sum()
print(
    "Hourly maximum greater than weekly total:",
    rainfall_consistency_errors
)

Hourly maximum greater than weekly total: 0


Check duplicate weekly geographic keys

In [25]:
duplicate_weather_keys = weather.duplicated(
    subset=[
        "year",
        "week",
        "reporting_area"
    ]
).sum()
print(
    "Duplicate year week reporting area keys:",
    duplicate_weather_keys
)

Duplicate year week reporting area keys: 0


Check number of reporting areas

In [26]:
print(
    "Reporting areas:",
    weather["reporting_area"].nunique()
)
print(
    "Administrative districts:",
    weather[
        "administrative_district"
    ].nunique()
)

Reporting areas: 26
Administrative districts: 25


Check reporting areas per year

In [28]:
weather_areas_per_year = (
    weather
    .groupby("year")["reporting_area"]
    .nunique()
)
weather_areas_per_year
print(
    "Minimum areas per year:",
    weather_areas_per_year.min()
)
print(
    "Maximum areas per year:",
    weather_areas_per_year.max()
)

Minimum areas per year: 26
Maximum areas per year: 26


Check represented weeks per year

In [29]:
weather_weeks_per_year = (
    weather
    .groupby("year")["week"]
    .nunique()
)
weather_weeks_per_year

year
2010    52
2011    52
2012    52
2013    52
2014    52
2015    52
2016    53
2017    52
2018    52
2019    52
2020    52
2021    53
2022    52
2023    52
2024    52
Name: week, dtype: int64

Check geographic completeness of every represented week

In [30]:
areas_per_week = (
    weather
    .groupby(
        ["year", "week"]
    )["reporting_area"]
    .nunique()
)
areas_per_week.value_counts().sort_index()

reporting_area
26    782
Name: count, dtype: int64

In [31]:
incomplete_weeks = areas_per_week[
    areas_per_week != 26
]
print(
    "Incomplete represented weeks:",
    len(incomplete_weeks)
)
incomplete_weeks.head()

Incomplete represented weeks: 0


Series([], Name: reporting_area, dtype: int64)

Build the actual weather calendar

In [32]:
weather_calendar = (
    weather[
        [
            "year",
            "week",
            "start_date",
            "end_date"
        ]
    ]
    .drop_duplicates()
    .sort_values("start_date")
    .reset_index(drop=True)
)
print(
    "Unique weather reporting periods:",
    len(weather_calendar)
)
weather_calendar.head()

Unique weather reporting periods: 782


,year,week,start_date,end_date
0,2010,2,2010-01-02,2010-01-08
1,2010,3,2010-01-09,2010-01-15
2,2010,4,2010-01-16,2010-01-22
3,2010,5,2010-01-23,2010-01-29
4,2010,6,2010-01-30,2010-02-05


In [33]:
weather_calendar.tail()

,year,week,start_date,end_date
777,2024,48,2024-11-23,2024-11-29
778,2024,49,2024-11-30,2024-12-06
779,2024,50,2024-12-07,2024-12-13
780,2024,51,2024-12-14,2024-12-20
781,2024,52,2024-12-21,2024-12-27


Check date consistency within each week

In [34]:
weather_date_consistency = (
    weather
    .groupby(
        ["year", "week"]
    )
    .agg(
        unique_start_dates=(
            "start_date",
            "nunique"
        ),
        unique_end_dates=(
            "end_date",
            "nunique"
        )
    )
)
weather_inconsistent_dates = (
    weather_date_consistency[
        (
            weather_date_consistency[
                "unique_start_dates"
            ] != 1
        )
        |
        (
            weather_date_consistency[
                "unique_end_dates"
            ] != 1
        )
    ]
)
print(
    "Weeks with inconsistent dates across areas:",
    len(weather_inconsistent_dates)
)

Weeks with inconsistent dates across areas: 0


Check chronological continuity

In [35]:
weather_calendar[
    "days_from_previous_week"
] = (
    weather_calendar["start_date"]
    .diff()
    .dt.days
)
weather_calendar[
    "days_from_previous_week"
].value_counts().sort_index()

days_from_previous_week
7.0    781
Name: count, dtype: int64

In [36]:
weather_large_gaps = weather_calendar[
    weather_calendar[
        "days_from_previous_week"
    ] > 7
]
print(
    "Chronological gaps greater than 7 days:",
    len(weather_large_gaps)
)

Chronological gaps greater than 7 days: 0


In [37]:
weather_overlaps = weather_calendar[
    weather_calendar[
        "days_from_previous_week"
    ] < 7
]
print(
    "Overlapping/short interval transitions:",
    len(weather_overlaps)
)

Overlapping/short interval transitions: 0


Check overall date coverage

In [39]:
print(
    "First weather period:",
    weather_calendar[
        "start_date"
    ].min(),
    "to",
    weather_calendar[
        "end_date"
    ].iloc[
        weather_calendar[
            "start_date"
        ].argmin()
    ]
)
print(
    "Last weather period:",
    weather_calendar[
        "start_date"
    ].max(),
    "to",
    weather_calendar[
        "end_date"
    ].iloc[
        weather_calendar[
            "start_date"
        ].argmax()
    ]
)
weather_calendar.tail(3)

First weather period: 2010-01-02 00:00:00 to 2010-01-08 00:00:00
Last weather period: 2024-12-21 00:00:00 to 2024-12-27 00:00:00


,year,week,start_date,end_date,days_from_previous_week
779,2024,50,2024-12-07,2024-12-13,7.0
780,2024,51,2024-12-14,2024-12-20,7.0
781,2024,52,2024-12-21,2024-12-27,7.0


Identify weeks crossing month boundaries

In [40]:
weather["crosses_month"] = (
    weather["start_date"].dt.to_period("M") !=
    weather["end_date"].dt.to_period("M")
)
cross_month_count = (
    weather["crosses_month"].sum()
)
cross_month_percentage = (
    weather["crosses_month"].mean() * 100
)
print(
    "Weather observations crossing month boundaries:",
    cross_month_count
)
print(
    "Percentage crossing month boundaries:",
    round(
        cross_month_percentage,
        2
    ),
    "%"
)

Weather observations crossing month boundaries: 4030
Percentage crossing month boundaries: 19.82 %


Identify year-boundary observations

In [41]:
weather["crosses_year"] = (
    weather["start_date"].dt.year !=
    weather["end_date"].dt.year
)
print(
    "Weather observations crossing year boundaries:",
    weather["crosses_year"].sum()
)

Weather observations crossing year boundaries: 312


Inspect crossing month examples

In [42]:
weather.loc[
    weather["crosses_month"],
    [
        "year",
        "week",
        "reporting_area",
        "start_date",
        "end_date",
        "temperature_mean_c",
        "humidity_mean_percent",
        "rainfall_total_mm",
        "rainy_days",
        "wind_speed_mean_kmh"
    ]
].head(20)

,year,week,reporting_area,start_date,end_date,temperature_mean_c,humidity_mean_percent,rainfall_total_mm,rainy_days,wind_speed_mean_kmh
104,2010,6,Ampara,2010-01-30,2010-02-05,24.97,84.53,69.6,6,12.80
105,2010,6,Anuradhapura,2010-01-30,2010-02-05,24.98,75.10,2.0,2,11.68
106,2010,6,Badulla,2010-01-30,2010-02-05,22.30,88.94,58.4,4,5.83
107,2010,6,Batticaloa,2010-01-30,2010-02-05,25.27,84.44,39.6,6,11.15
108,2010,6,Colombo,2010-01-30,2010-02-05,26.72,74.01,29.3,4,12.24
109,2010,6,Galle,2010-01-30,2010-02-05,26.23,84.52,92.6,6,5.15
110,2010,6,Gampaha,2010-01-30,2010-02-05,26.83,71.52,29.3,4,12.24
111,2010,6,Hambantota,2010-01-30,2010-02-05,26.17,78.20,36.9,3,12.03
112,2010,6,Jaffna,2010-01-30,2010-02-05,25.77,67.82,0.0,0,23.80
113,2010,6,Kalmunai,2010-01-30,2010-02-05,25.63,81.42,73.0,6,16.41


## 2. Variable-Specific Monthly Aggregation Strategy

Weather variables cannot all be aggregated using the same mathematical operation.

### Mean state variables

The following variables represent average environmental conditions:

- temperature mean
- humidity mean
- wind speed mean

For these variables, monthly values will be calculated using day weighted means based on the number of days from each weekly reporting period that overlap the calendar month.

### Accumulated variables

Weekly total rainfall represents an accumulated quantity. For weeks crossing a month boundary, the weekly total will be allocated proportionally according to the number of reporting days falling within each month, after which the allocated values will be summed.

Rainy days are also an accumulated weekly quantity. Because the exact dates of rainy days are unavailable in the weekly summary dataset, crossing week rainy day counts cannot be assigned exactly to individual months. A day weighted proportional allocation will therefore be used and explicitly treated as an approximation.

### Minimum and maximum variables

Weekly minimum and maximum temperature, humidity, wind speed and maximum hourly rainfall represent extrema rather than totals or means. These values cannot be divided proportionally across calendar months.

Monthly extrema will therefore be derived from the weekly extrema associated with reporting periods overlapping each month. For weeks crossing a month boundary, the exact day on which an extreme occurred is unknown. Consequently, monthly extrema near calendar boundaries should be interpreted as approximations derived from weekly summaries.

### Important limitation

The source dataset contains weekly weather summaries rather than daily weather observations. Therefore, exact reconstruction of daily weather conditions within a crossing week is not possible. Temporal allocation of weekly summaries requires explicit assumptions, which are documented rather than treated as observed daily values.

## 3. Weekly Weather Validation Summary

The standardized weekly weather dataset contains 20,332 observations covering 26 reporting areas and 25 corresponding administrative districts.

All reporting periods contain seven days. No invalid dates, standard missing values, common sentinel missing value codes, duplicate year week reporting area keys, negative rainfall values, negative rainy-day values, negative wind-speed values, temperature ordering errors, humidity ordering errors, wind ordering errors or rainfall consistency errors were identified.

All represented weeks contain all 26 reporting areas. The dataset contains 782 consecutive weekly reporting periods, and every consecutive reporting period start date is separated by exactly seven days. Therefore, no chronological weekly gaps or overlaps were identified.

The weather data cover the period from 2 January 2010 to 27 December 2024.

A total of 4,030 reporting-area observations (19.82%) cross calendar-month boundaries, while 312 observations cross calendar year boundaries. Therefore, monthly weather construction is based on the validated reporting period dates rather than the source year and week labels.

In [43]:
mean_variables = [
    "temperature_mean_c",
    "humidity_mean_percent",
    "wind_speed_mean_kmh"
]
total_variables = [
    "rainfall_total_mm",
    "rainy_days"
]
minimum_variables = [
    "temperature_min_c",
    "humidity_min_percent"
]
maximum_variables = [
    "temperature_max_c",
    "humidity_max_percent",
    "rainfall_max_hourly_mm",
    "wind_speed_max_kmh"
]
print("Mean variables:", mean_variables)
print("Total variables:", total_variables)
print("Minimum variables:", minimum_variables)
print("Maximum variables:", maximum_variables)

Mean variables: ['temperature_mean_c', 'humidity_mean_percent', 'wind_speed_mean_kmh']
Total variables: ['rainfall_total_mm', 'rainy_days']
Minimum variables: ['temperature_min_c', 'humidity_min_percent']
Maximum variables: ['temperature_max_c', 'humidity_max_percent', 'rainfall_max_hourly_mm', 'wind_speed_max_kmh']


## 4. Temporal Allocation of Weekly Weather Summaries

Weekly weather summaries are expanded into temporary daily proxy records to support calendar month aggregation.

For weekly mean variables, the weekly mean is associated with each reporting day. Monthly means are then calculated from these day-weighted contributions.

For accumulated variables such as rainfall total and rainy days, the weekly quantity is divided equally across the seven reporting days before monthly summation.

Weekly minimum and maximum variables are not divided. Their weekly extrema are associated with each overlapping calendar month and subsequently aggregated using minimum or maximum operations.

The temporary daily representation is an allocation mechanism rather than reconstructed daily meteorological observations.

Create the daily records

In [44]:
daily_weather_records = []
for _, row in weather.iterrows():
    dates = pd.date_range(
        start=row["start_date"],
        end=row["end_date"],
        freq="D"
    )
    number_of_days = len(dates)
    for date in dates:
        daily_weather_records.append({
            "date": date,
            "reporting_area":
                row["reporting_area"],
            "administrative_district":
                row["administrative_district"],
            # Mean-state variables
            "temperature_mean_c":
                row["temperature_mean_c"],
            "humidity_mean_percent":
                row["humidity_mean_percent"],
            "wind_speed_mean_kmh":
                row["wind_speed_mean_kmh"],
            # Proportionally allocated totals
            "rainfall_allocated_mm":
                row["rainfall_total_mm"] /
                number_of_days,
            "rainy_days_allocated":
                row["rainy_days"] /
                number_of_days,
            # Weekly extrema
            "temperature_min_c":
                row["temperature_min_c"],
            "temperature_max_c":
                row["temperature_max_c"],
            "humidity_min_percent":
                row["humidity_min_percent"],
            "humidity_max_percent":
                row["humidity_max_percent"],
            "rainfall_max_hourly_mm":
                row["rainfall_max_hourly_mm"],
            "wind_speed_max_kmh":
                row["wind_speed_max_kmh"]
        })

In [45]:
daily_weather = pd.DataFrame(
    daily_weather_records
)
print(
    "Daily weather rows:",
    len(daily_weather)
)
daily_weather.head()

Daily weather rows: 142324


,date,reporting_area,administrative_district,temperature_mean_c,humidity_mean_percent,wind_speed_mean_kmh,rainfall_allocated_mm,rainy_days_allocated,temperature_min_c,temperature_max_c,humidity_min_percent,humidity_max_percent,rainfall_max_hourly_mm,wind_speed_max_kmh
0,2010-01-02,Ampara,Ampara,25.06,77.1,12.47,0.828571,0.571429,21.9,28.6,59,91,0.6,21.1
1,2010-01-03,Ampara,Ampara,25.06,77.1,12.47,0.828571,0.571429,21.9,28.6,59,91,0.6,21.1
2,2010-01-04,Ampara,Ampara,25.06,77.1,12.47,0.828571,0.571429,21.9,28.6,59,91,0.6,21.1
3,2010-01-05,Ampara,Ampara,25.06,77.1,12.47,0.828571,0.571429,21.9,28.6,59,91,0.6,21.1
4,2010-01-06,Ampara,Ampara,25.06,77.1,12.47,0.828571,0.571429,21.9,28.6,59,91,0.6,21.1


Validate expected number of daily rows

In [46]:
expected_daily_rows = (
    len(weather) * 7
)
print(
    "Expected daily rows:",
    expected_daily_rows
)
print(
    "Actual daily rows:",
    len(daily_weather)
)
print(
    "Daily row count correct:",
    expected_daily_rows ==
    len(daily_weather)
)

Expected daily rows: 142324
Actual daily rows: 142324
Daily row count correct: True


Check the temporary daily date range

In [47]:
print(
    "First daily date:",
    daily_weather["date"].min()
)
print(
    "Last daily date:",
    daily_weather["date"].max()
)
print(
    "Unique calendar dates:",
    daily_weather["date"].nunique()
)

First daily date: 2010-01-02 00:00:00
Last daily date: 2024-12-27 00:00:00
Unique calendar dates: 5474


Create the calendar month

In [48]:
daily_weather["month"] = (
    daily_weather["date"]
    .dt.to_period("M")
)

In [49]:
print(
    "First month:",
    daily_weather["month"].min()
)
print(
    "Last month:",
    daily_weather["month"].max()
)
print(
    "Unique months:",
    daily_weather["month"].nunique()
)

First month: 2010-01
Last month: 2024-12
Unique months: 180


In [50]:
days_per_month = (
    daily_weather
    .groupby(
        ["reporting_area", "month"]
    )["date"]
    .nunique()
    .reset_index(
        name="represented_days"
    )
)
days_per_month[
    "represented_days"
].value_counts().sort_index()

represented_days
27      26
28     286
29     104
30    1586
31    2678
Name: count, dtype: int64

In [51]:
partial_months = days_per_month[
    days_per_month[
        "represented_days"
    ] !=
    days_per_month["month"].dt.days_in_month
]
print(
    "Partial reporting-area months:",
    len(partial_months)
)
partial_months.head(30)

Partial reporting-area months: 52


,reporting_area,month,represented_days
0,Ampara,2010-01,30
179,Ampara,2024-12,27
180,Anuradhapura,2010-01,30
359,Anuradhapura,2024-12,27
360,Badulla,2010-01,30
539,Badulla,2024-12,27
540,Batticaloa,2010-01,30
719,Batticaloa,2024-12,27
720,Colombo,2010-01,30
899,Colombo,2024-12,27


Check which calendar months are partial

In [52]:
partial_calendar_months = (
    partial_months[
        [
            "month",
            "represented_days"
        ]
    ]
    .drop_duplicates()
    .sort_values("month")
)
partial_calendar_months

,month,represented_days
0,2010-01,30
179,2024-12,27


Aggregate the mean state variables

In [53]:
monthly_means = (
    daily_weather
    .groupby(
        ["reporting_area", "month"],
        as_index=False
    )[
        [
            "temperature_mean_c",
            "humidity_mean_percent",
            "wind_speed_mean_kmh"
        ]
    ]
    .mean()
)
monthly_means.head()

,reporting_area,month,temperature_mean_c,humidity_mean_percent,wind_speed_mean_kmh
0,Ampara,2010-01,25.299000,81.550333,11.141000
1,Ampara,2010-02,25.825714,79.899643,10.090000
2,Ampara,2010-03,27.562581,76.425806,8.334839
3,Ampara,2010-04,28.322000,80.794667,6.608333
4,Ampara,2010-05,28.722581,80.573548,8.302581


Aggregate rainfall and rainy days

In [54]:
monthly_totals = (
    daily_weather
    .groupby(
        ["reporting_area", "month"],
        as_index=False
    )[
        [
            "rainfall_allocated_mm",
            "rainy_days_allocated"
        ]
    ]
    .sum()
)

In [55]:
monthly_minima = (
    daily_weather
    .groupby(
        ["reporting_area", "month"],
        as_index=False
    )[
        [
            "temperature_min_c",
            "humidity_min_percent"
        ]
    ]
    .min()
)
monthly_minima.head()

,reporting_area,month,temperature_min_c,humidity_min_percent
0,Ampara,2010-01,21.7,59
1,Ampara,2010-02,21.8,53
2,Ampara,2010-03,22.1,49
3,Ampara,2010-04,24.7,54
4,Ampara,2010-05,25.1,57


In [56]:
monthly_maxima = (
    daily_weather
    .groupby(
        ["reporting_area", "month"],
        as_index=False
    )[
        [
            "temperature_max_c",
            "humidity_max_percent",
            "rainfall_max_hourly_mm",
            "wind_speed_max_kmh"
        ]
    ]
    .max()
)

monthly_maxima.head()

,reporting_area,month,temperature_max_c,humidity_max_percent,rainfall_max_hourly_mm,wind_speed_max_kmh
0,Ampara,2010-01,29.1,97,5.7,22.5
1,Ampara,2010-02,31.9,95,5.7,22.5
2,Ampara,2010-03,33.1,96,2.5,18.9
3,Ampara,2010-04,33.1,97,3.8,14.2
4,Ampara,2010-05,34.4,96,6.6,28.6


Combine all monthly weather components

In [57]:
monthly_weather_26 = (
    monthly_means
    .merge(
        monthly_totals,
        on=["reporting_area", "month"],
        how="outer"
    )
    .merge(
        monthly_minima,
        on=["reporting_area", "month"],
        how="outer"
    )
    .merge(
        monthly_maxima,
        on=["reporting_area", "month"],
        how="outer"
    )
)

monthly_weather_26.head()

,reporting_area,month,temperature_mean_c,humidity_mean_percent,wind_speed_mean_kmh,rainfall_allocated_mm,rainy_days_allocated,temperature_min_c,humidity_min_percent,temperature_max_c,humidity_max_percent,rainfall_max_hourly_mm,wind_speed_max_kmh
0,Ampara,2010-01,25.299000,81.550333,11.141000,81.885714,22.714286,21.7,59,29.1,97,5.7,22.5
1,Ampara,2010-02,25.825714,79.899643,10.090000,69.942857,12.571429,21.8,53,31.9,95,5.7,22.5
2,Ampara,2010-03,27.562581,76.425806,8.334839,29.300000,16.000000,22.1,49,33.1,96,2.5,18.9
3,Ampara,2010-04,28.322000,80.794667,6.608333,82.871429,28.714286,24.7,54,33.1,97,3.8,14.2
4,Ampara,2010-05,28.722581,80.573548,8.302581,92.057143,27.000000,25.1,57,34.4,96,6.6,28.6


In [58]:
print(
    "Monthly weather rows:",
    len(monthly_weather_26)
)
print(
    "Reporting areas:",
    monthly_weather_26[
        "reporting_area"
    ].nunique()
)
print(
    "Months:",
    monthly_weather_26[
        "month"
    ].nunique()
)

Monthly weather rows: 4680
Reporting areas: 26
Months: 180


In [59]:
weather_months_per_area = (
    monthly_weather_26
    .groupby(
        "reporting_area"
    )["month"]
    .nunique()
    .sort_values()
)
print(
    "Minimum months per reporting area:",
    weather_months_per_area.min()
)
print(
    "Maximum months per reporting area:",
    weather_months_per_area.max()
)

Minimum months per reporting area: 180
Maximum months per reporting area: 180


Check missing monthly values

In [63]:
monthly_weather_26.isna().sum()

reporting_area            0
month                     0
temperature_mean_c        0
humidity_mean_percent     0
wind_speed_mean_kmh       0
rainfall_allocated_mm     0
rainy_days_allocated      0
temperature_min_c         0
humidity_min_percent      0
temperature_max_c         0
humidity_max_percent      0
rainfall_max_hourly_mm    0
wind_speed_max_kmh        0
dtype: int64

In [65]:
duplicate_monthly_weather_keys = (
    monthly_weather_26
    .duplicated(
        subset=[
            "reporting_area",
            "month"
        ]
    )
    .sum()
)
print(
    "Duplicate reporting-area-month keys:",
    duplicate_monthly_weather_keys
)

Duplicate reporting-area-month keys: 0


In [69]:
monthly_totals = (
    daily_weather
    .groupby(
        ["reporting_area", "month"],
        as_index=False
    )[
        [
            "rainfall_allocated_mm",
            "rainy_days_allocated"
        ]
    ]
    .sum()
)

In [70]:
monthly_totals = monthly_totals.rename(
    columns={
        "rainfall_allocated_mm": "rainfall_total_mm",
        "rainy_days_allocated": "rainy_days"
    }
)
monthly_totals.head()

,reporting_area,month,rainfall_total_mm,rainy_days
0,Ampara,2010-01,81.885714,22.714286
1,Ampara,2010-02,69.942857,12.571429
2,Ampara,2010-03,29.300000,16.000000
3,Ampara,2010-04,82.871429,28.714286
4,Ampara,2010-05,92.057143,27.000000


In [71]:
monthly_weather_26 = (
    monthly_means
    .merge(
        monthly_totals,
        on=["reporting_area", "month"],
        how="outer"
    )
    .merge(
        monthly_minima,
        on=["reporting_area", "month"],
        how="outer"
    )
    .merge(
        monthly_maxima,
        on=["reporting_area", "month"],
        how="outer"
    )
)

monthly_weather_26.head()

,reporting_area,month,temperature_mean_c,humidity_mean_percent,wind_speed_mean_kmh,rainfall_total_mm,rainy_days,temperature_min_c,humidity_min_percent,temperature_max_c,humidity_max_percent,rainfall_max_hourly_mm,wind_speed_max_kmh
0,Ampara,2010-01,25.299000,81.550333,11.141000,81.885714,22.714286,21.7,59,29.1,97,5.7,22.5
1,Ampara,2010-02,25.825714,79.899643,10.090000,69.942857,12.571429,21.8,53,31.9,95,5.7,22.5
2,Ampara,2010-03,27.562581,76.425806,8.334839,29.300000,16.000000,22.1,49,33.1,96,2.5,18.9
3,Ampara,2010-04,28.322000,80.794667,6.608333,82.871429,28.714286,24.7,54,33.1,97,3.8,14.2
4,Ampara,2010-05,28.722581,80.573548,8.302581,92.057143,27.000000,25.1,57,34.4,96,6.6,28.6


In [72]:
original_rainfall_total = (
    weather["rainfall_total_mm"].sum()
)
monthly_rainfall_total = (
    monthly_weather_26["rainfall_total_mm"].sum()
)
print(
    "Original weekly rainfall total:",
    original_rainfall_total
)
print(
    "Monthly allocated rainfall total:",
    monthly_rainfall_total
)
print(
    "Rainfall preserved:",
    np.isclose(
        original_rainfall_total,
        monthly_rainfall_total
    )
)

Original weekly rainfall total: 779174.1000000001
Monthly allocated rainfall total: 779174.1
Rainfall preserved: True


In [73]:
original_rainy_days = (
    weather["rainy_days"].sum()
)
monthly_rainy_days = (
    monthly_weather_26[
        "rainy_days"
    ].sum()
)
print(
    "Original weekly rainy-day total:",
    original_rainy_days
)
print(
    "Monthly allocated rainy-day total:",
    monthly_rainy_days
)
print(
    "Rainy days preserved:",
    np.isclose(
        original_rainy_days,
        monthly_rainy_days
    )
)

Original weekly rainy-day total: 111765
Monthly allocated rainy-day total: 111765.0
Rainy days preserved: True


In [74]:
monthly_temperature_errors = (
    (
        monthly_weather_26[
            "temperature_min_c"
        ] >
        monthly_weather_26[
            "temperature_mean_c"
        ]
    )
    |
    (
        monthly_weather_26[
            "temperature_mean_c"
        ] >
        monthly_weather_26[
            "temperature_max_c"
        ]
    )
).sum()

print(
    "Monthly temperature ordering errors:",
    monthly_temperature_errors
)

Monthly temperature ordering errors: 0


Validate monthly humidity consistency

In [75]:
monthly_humidity_errors = (
    (
        monthly_weather_26[
            "humidity_min_percent"
        ] >
        monthly_weather_26[
            "humidity_mean_percent"
        ]
    )
    |
    (
        monthly_weather_26[
            "humidity_mean_percent"
        ] >
        monthly_weather_26[
            "humidity_max_percent"
        ]
    )
).sum()

print(
    "Monthly humidity ordering errors:",
    monthly_humidity_errors
)

Monthly humidity ordering errors: 0


Validate monthly wind consistency

In [76]:
monthly_wind_errors = (
    monthly_weather_26[
        "wind_speed_mean_kmh"
    ] >
    monthly_weather_26[
        "wind_speed_max_kmh"
    ]
).sum()

print(
    "Monthly wind ordering errors:",
    monthly_wind_errors
)

Monthly wind ordering errors: 0


Check monthly rainfall values

In [77]:
negative_monthly_rainfall = (
    monthly_weather_26[
        "rainfall_total_mm"
    ] < 0
).sum()
negative_monthly_rainy_days = (
    monthly_weather_26[
        "rainy_days"
    ] < 0
).sum()
print(
    "Negative monthly rainfall values:",
    negative_monthly_rainfall
)
print(
    "Negative monthly rainy-day values:",
    negative_monthly_rainy_days
)

Negative monthly rainfall values: 0
Negative monthly rainy-day values: 0


Inspect Ampara and Kalmunai coordinates

In [78]:
ampara_kalmunai_locations = (
    weather[
        weather["reporting_area"].isin(
            ["Ampara", "Kalmunai"]
        )
    ][
        [
            "reporting_area",
            "latitude",
            "longitude",
            "administrative_district"
        ]
    ]
    .drop_duplicates()
    .sort_values("reporting_area")
)

ampara_kalmunai_locations

,reporting_area,latitude,longitude,administrative_district
0,Ampara,7.29,81.67,Ampara
9,Kalmunai,7.41,81.83,Ampara


In [79]:
coordinate_counts = (
    weather[
        weather["reporting_area"].isin(
            ["Ampara", "Kalmunai"]
        )
    ]
    .groupby("reporting_area")
    .agg(
        unique_latitudes=(
            "latitude",
            "nunique"
        ),
        unique_longitudes=(
            "longitude",
            "nunique"
        )
    )
)

coordinate_counts

,unique_latitudes,unique_longitudes
reporting_area,,
Ampara,1,1
Kalmunai,1,1


Compare Ampara and Kalmunai monthly weather

In [81]:
ampara_kalmunai_monthly = (
    monthly_weather_26[
        monthly_weather_26[
            "reporting_area"
        ].isin(
            ["Ampara", "Kalmunai"]
        )
    ]
    .sort_values(
        ["month", "reporting_area"]
    )
)

ampara_kalmunai_monthly.head(20)
print(
    ampara_kalmunai_monthly
    .groupby("reporting_area")["month"]
    .nunique()
)

reporting_area
Ampara      180
Kalmunai    180
Name: month, dtype: int64


In [82]:
ampara_kalmunai_comparison = (
    ampara_kalmunai_monthly
    .pivot(
        index="month",
        columns="reporting_area",
        values=[
            "temperature_mean_c",
            "humidity_mean_percent",
            "rainfall_total_mm",
            "wind_speed_mean_kmh"
        ]
    )
)
ampara_kalmunai_comparison.head()

temperature_mean_c            humidity_mean_percent             \
reporting_area             Ampara   Kalmunai                Ampara   Kalmunai   
month                                                                           
2010-01                 25.299000  25.900667             81.550333  78.991000   
2010-02                 25.825714  26.426071             79.899643  77.197857   
2010-03                 27.562581  27.941613             76.425806  75.267742   
2010-04                 28.322000  28.779667             80.794667  78.616333   
2010-05                 28.722581  29.229355             80.573548  77.910323   

               rainfall_total_mm            wind_speed_mean_kmh             
reporting_area            Ampara   Kalmunai              Ampara   Kalmunai  
month                                                                       
2010-01                81.885714  94.857143           11.141000  14.261000  
2010-02                69.942857  74.200000           10.090000  12.971429  
2010-03                29.300000  22.471429            8.334839  10.975161  
2010-04                82.871429  64.971429            6.608333   9.121333  
2010-05                92.057143  66.828571            8.302581   8.212258

In [83]:
variables_to_compare = [
    "temperature_mean_c",
    "humidity_mean_percent",
    "rainfall_total_mm",
    "wind_speed_mean_kmh"
]
for variable in variables_to_compare:

    comparison = (
        monthly_weather_26[
            monthly_weather_26[
                "reporting_area"
            ].isin(
                ["Ampara", "Kalmunai"]
            )
        ]
        .pivot(
            index="month",
            columns="reporting_area",
            values=variable
        )
    )

    correlation = (
        comparison["Ampara"]
        .corr(comparison["Kalmunai"])
    )

    print(
        variable,
        "correlation:",
        round(correlation, 4)
    )

temperature_mean_c correlation: 0.9842
humidity_mean_percent correlation: 0.9348
rainfall_total_mm correlation: 0.9603
wind_speed_mean_kmh correlation: 0.7472


## 5. Conversion from Reporting Areas to Administrative Districts

The weather dataset contains 26 reporting areas, whereas the final dengue forecasting dataset uses 25 administrative districts.

Ampara and Kalmunai are represented by separate weather locations but both correspond to the Ampara administrative district. Their coordinates remain constant throughout the study period.

Monthly comparison showed strong correlations between the two locations for mean temperature, humidity and rainfall, while wind speed showed a lower but still positive correlation.

Weather measurements are not additive. Therefore, Ampara and Kalmunai weather values must not be summed when constructing administrative district weather features.

For the final 25-district dataset, an unweighted arithmetic mean of the monthly Ampara and Kalmunai weather summaries is used to represent the Ampara administrative district. All other reporting areas have a one to one correspondence with an administrative district and therefore retain their monthly weather values unchanged.

This approach provides a transparent spatial aggregation rule when more detailed area-weighting information is unavailable. The resulting Ampara weather features should be interpreted as representative district level environmental summaries rather than exact spatial averages over the administrative district.

Restore administrative-district mapping

In [84]:
reporting_admin_mapping = (
    weather[
        [
            "reporting_area",
            "administrative_district"
        ]
    ]
    .drop_duplicates()
)
reporting_admin_mapping.head()

,reporting_area,administrative_district
0,Ampara,Ampara
1,Anuradhapura,Anuradhapura
2,Badulla,Badulla
3,Batticaloa,Batticaloa
4,Colombo,Colombo


In [85]:
print(
    "Reporting-area mappings:",
    len(reporting_admin_mapping)
)
print(
    "Unique reporting areas:",
    reporting_admin_mapping[
        "reporting_area"
    ].nunique()
)
print(
    "Unique administrative districts:",
    reporting_admin_mapping[
        "administrative_district"
    ].nunique()
)

Reporting-area mappings: 26
Unique reporting areas: 26
Unique administrative districts: 25


Attach administrative districts

In [86]:
monthly_weather_mapped = (
    monthly_weather_26
    .merge(
        reporting_admin_mapping,
        on="reporting_area",
        how="left"
    )
)
monthly_weather_mapped.head()

,reporting_area,month,temperature_mean_c,humidity_mean_percent,wind_speed_mean_kmh,rainfall_total_mm,rainy_days,temperature_min_c,humidity_min_percent,temperature_max_c,humidity_max_percent,rainfall_max_hourly_mm,wind_speed_max_kmh,administrative_district
0,Ampara,2010-01,25.299000,81.550333,11.141000,81.885714,22.714286,21.7,59,29.1,97,5.7,22.5,Ampara
1,Ampara,2010-02,25.825714,79.899643,10.090000,69.942857,12.571429,21.8,53,31.9,95,5.7,22.5,Ampara
2,Ampara,2010-03,27.562581,76.425806,8.334839,29.300000,16.000000,22.1,49,33.1,96,2.5,18.9,Ampara
3,Ampara,2010-04,28.322000,80.794667,6.608333,82.871429,28.714286,24.7,54,33.1,97,3.8,14.2,Ampara
4,Ampara,2010-05,28.722581,80.573548,8.302581,92.057143,27.000000,25.1,57,34.4,96,6.6,28.6,Ampara


In [87]:
print(
    "Unmapped administrative districts:",
    monthly_weather_mapped[
        "administrative_district"
    ].isna().sum()
)

Unmapped administrative districts: 0


In [88]:
weather_feature_columns = [
    "temperature_mean_c",
    "humidity_mean_percent",
    "wind_speed_mean_kmh",
    "rainfall_total_mm",
    "rainy_days",
    "temperature_min_c",
    "humidity_min_percent",
    "temperature_max_c",
    "humidity_max_percent",
    "rainfall_max_hourly_mm",
    "wind_speed_max_kmh"
]

In [89]:
monthly_weather_25 = (
    monthly_weather_mapped
    .groupby(
        [
            "administrative_district",
            "month"
        ],
        as_index=False
    )[weather_feature_columns]
    .mean()
)

In [90]:
print(
    "Administrative districts:",
    monthly_weather_25[
        "administrative_district"
    ].nunique()
)
print(
    "Months:",
    monthly_weather_25[
        "month"
    ].nunique()
)
print(
    "Total rows:",
    len(monthly_weather_25)
)

Administrative districts: 25
Months: 180
Total rows: 4500


Validate months per district

In [91]:
months_per_weather_district = (
    monthly_weather_25
    .groupby(
        "administrative_district"
    )["month"]
    .nunique()
)
print(
    "Minimum months per district:",
    months_per_weather_district.min()
)
print(
    "Maximum months per district:",
    months_per_weather_district.max()
)

Minimum months per district: 180
Maximum months per district: 180


In [92]:
duplicate_district_months = (
    monthly_weather_25
    .duplicated(
        subset=[
            "administrative_district",
            "month"
        ]
    )
    .sum()
)
print(
    "Duplicate district-month keys:",
    duplicate_district_months
)

Duplicate district-month keys: 0


In [93]:
monthly_weather_25.isna().sum()

administrative_district    0
month                      0
temperature_mean_c         0
humidity_mean_percent      0
wind_speed_mean_kmh        0
rainfall_total_mm          0
rainy_days                 0
temperature_min_c          0
humidity_min_percent       0
temperature_max_c          0
humidity_max_percent       0
rainfall_max_hourly_mm     0
wind_speed_max_kmh         0
dtype: int64

In [ ]:
colombo_reporting = (
    monthly_weather_26[
        monthly_weather_26[
            "reporting_area"
        ] == "Colombo"
    ]
    .set_index("month")[
        weather_feature_columns
    ]
)
colombo_admin = (
    monthly_weather_25[
        monthly_weather_25[
            "administrative_district"
        ] == "Colombo"
    ]
    .set_index("month")[
        weather_feature_columns
    ]
)
print(
    "Colombo unchanged:",
    np.allclose(
        colombo_reporting.values,
        colombo_admin.values
    )
)

Colombo unchanged: True


## Boundary-Month Coverage

The weekly weather reporting sequence is chronologically continuous; however, the available study window begins on 2 January 2010 and ends on 27 December 2024.

Consequently:

- January 2010 contains 30 represented weather days rather than 31.
- December 2024 contains 27 represented weather days rather than 31.

These are dataset-boundary effects rather than internal missing-week gaps.

The boundary months are retained in the prepared dataset for traceability, but their incomplete calendar coverage is explicitly recorded. The final modeling stage will determine whether these boundary observations should be excluded to ensure that only complete calendar months are used for model training and evaluation.

In [95]:
monthly_weather_25[
    "complete_weather_month"
] = True
monthly_weather_25.loc[
    monthly_weather_25["month"].isin(
        [
            pd.Period("2010-01"),
            pd.Period("2024-12")
        ]
    ),
    "complete_weather_month"
] = False

In [96]:
monthly_weather_25[
    ~monthly_weather_25[
        "complete_weather_month"
    ]
][
    [
        "administrative_district",
        "month",
        "complete_weather_month"
    ]
].head(30)

,administrative_district,month,complete_weather_month
0,Ampara,2010-01,False
179,Ampara,2024-12,False
180,Anuradhapura,2010-01,False
359,Anuradhapura,2024-12,False
360,Badulla,2010-01,False
539,Badulla,2024-12,False
540,Batticaloa,2010-01,False
719,Batticaloa,2024-12,False
720,Colombo,2010-01,False
899,Colombo,2024-12,False


In [97]:
print(
    "Incomplete district-month weather observations:",
    (
        ~monthly_weather_25[
            "complete_weather_month"
        ]
    ).sum()
)

Incomplete district-month weather observations: 50


Validate Ampara aggregation

In [99]:
ampara_sources = (
    monthly_weather_mapped[
        monthly_weather_mapped[
            "administrative_district"
        ] == "Ampara"
    ]
)
expected_ampara = (
    ampara_sources
    .groupby("month")[
        weather_feature_columns
    ]
    .mean()
)
actual_ampara = (
    monthly_weather_25[
        monthly_weather_25[
            "administrative_district"
        ] == "Ampara"
    ]
    .set_index("month")[
        weather_feature_columns
    ]
)
ampara_spatial_aggregation_correct = (
    np.allclose(
        expected_ampara.values,
        actual_ampara.values
    )
)

print(
    "Ampara spatial aggregation correct:",
    ampara_spatial_aggregation_correct
)

Ampara spatial aggregation correct: True


Validate weather consistency after spatial averaging

In [100]:
admin_temperature_errors = (
    (
        monthly_weather_25["temperature_min_c"] >
        monthly_weather_25["temperature_mean_c"]
    )
    |
    (
        monthly_weather_25["temperature_mean_c"] >
        monthly_weather_25["temperature_max_c"]
    )
).sum()
admin_humidity_errors = (
    (
        monthly_weather_25["humidity_min_percent"] >
        monthly_weather_25["humidity_mean_percent"]
    )
    |
    (
        monthly_weather_25["humidity_mean_percent"] >
        monthly_weather_25["humidity_max_percent"]
    )
).sum()

admin_wind_errors = (
    monthly_weather_25["wind_speed_mean_kmh"] >
    monthly_weather_25["wind_speed_max_kmh"]
).sum()

print(
    "Temperature ordering errors:",
    admin_temperature_errors
)

print(
    "Humidity ordering errors:",
    admin_humidity_errors
)

print(
    "Wind ordering errors:",
    admin_wind_errors
)

Temperature ordering errors: 0
Humidity ordering errors: 0
Wind ordering errors: 0


In [101]:
monthly_weather_25["date"] = (
    monthly_weather_25["month"]
    .dt.to_timestamp()
)
monthly_weather_25["year"] = (
    monthly_weather_25["date"].dt.year
)
monthly_weather_25["month_number"] = (
    monthly_weather_25["date"].dt.month
)

In [102]:
monthly_weather_25[
    [
        "date",
        "year",
        "month_number",
        "administrative_district"
    ]
].head()

,date,year,month_number,administrative_district
0,2010-01-01,2010,1,Ampara
1,2010-02-01,2010,2,Ampara
2,2010-03-01,2010,3,Ampara
3,2010-04-01,2010,4,Ampara
4,2010-05-01,2010,5,Ampara


In [103]:
monthly_weather_final = (
    monthly_weather_25[
        [
            "date",
            "year",
            "month_number",
            "administrative_district",
            "temperature_mean_c",
            "temperature_min_c",
            "temperature_max_c",
            "humidity_mean_percent",
            "humidity_min_percent",
            "humidity_max_percent",
            "rainfall_total_mm",
            "rainfall_max_hourly_mm",
            "rainy_days",
            "wind_speed_mean_kmh",
            "wind_speed_max_kmh",
            "complete_weather_month"
        ]
    ]
    .sort_values(
        [
            "administrative_district",
            "date"
        ]
    )
    .reset_index(drop=True)
)

monthly_weather_final.head()

,date,year,month_number,administrative_district,temperature_mean_c,temperature_min_c,temperature_max_c,humidity_mean_percent,humidity_min_percent,humidity_max_percent,rainfall_total_mm,rainfall_max_hourly_mm,rainy_days,wind_speed_mean_kmh,wind_speed_max_kmh,complete_weather_month
0,2010-01-01,2010,1,Ampara,25.599833,22.20,28.95,80.270667,59.5,96.5,88.371429,4.35,21.714286,12.701000,24.15,False
1,2010-02-01,2010,2,Ampara,26.125893,22.20,31.40,78.548750,56.0,94.0,72.071429,4.35,12.571429,11.530714,24.15,True
2,2010-03-01,2010,3,Ampara,27.752097,22.55,32.55,75.846774,51.0,94.0,25.885714,2.45,14.500000,9.655000,21.65,True
3,2010-04-01,2010,4,Ampara,28.550833,25.20,32.80,79.705500,57.5,95.5,73.921429,3.45,28.214286,7.864833,16.35,True
4,2010-05-01,2010,5,Ampara,28.975968,25.45,34.20,79.241935,57.0,94.5,79.442857,4.95,28.000000,8.257419,28.35,True


In [ ]:
print(
    "Final monthly weather shape:",
    monthly_weather_final.shape
)
print(
    "Administrative districts:",
    monthly_weather_final[
        "administrative_district"
    ].nunique()
)
print(
    "Unique months:",
    monthly_weather_final[
        "date"
    ].nunique()
)
print(
    "Date range:",
    monthly_weather_final["date"].min(),
    "to",
    monthly_weather_final["date"].max()
)

Final monthly weather shape: (4500, 16)
Administrative districts: 25
Unique months: 180
Date range: 2010-01-01 00:00:00 to 2024-12-01 00:00:00


In [105]:
weather_validation = pd.DataFrame({
    "Validation Check": [
        "Administrative districts",
        "Calendar months",
        "District-month observations",
        "Minimum months per district",
        "Maximum months per district",
        "Duplicate district-month keys",
        "Missing weather feature values",
        "Temperature ordering errors",
        "Humidity ordering errors",
        "Wind ordering errors",
        "Ampara spatial aggregation correct",
        "Colombo unchanged",
        "Incomplete boundary district-months"
    ],

    "Result": [
        monthly_weather_final[
            "administrative_district"
        ].nunique(),

        monthly_weather_final[
            "date"
        ].nunique(),

        len(monthly_weather_final),

        months_per_weather_district.min(),

        months_per_weather_district.max(),

        duplicate_district_months,

        monthly_weather_final[
            weather_feature_columns
        ].isna().sum().sum(),

        admin_temperature_errors,

        admin_humidity_errors,

        admin_wind_errors,

        ampara_spatial_aggregation_correct,

        np.allclose(
            colombo_reporting.values,
            colombo_admin.values
        ),

        (
            ~monthly_weather_final[
                "complete_weather_month"
            ]
        ).sum()
    ]
})

weather_validation

,Validation Check,Result
0,Administrative districts,25
1,Calendar months,180
2,District-month observations,4500
3,Minimum months per district,180
4,Maximum months per district,180
5,Duplicate district-month keys,0
6,Missing weather feature values,0
7,Temperature ordering errors,0
8,Humidity ordering errors,0
9,Wind ordering errors,0


In [106]:
weather_output = (
    INTERIM_DIR /
    "weather_monthly_prepared_2010_2024.csv"
)
monthly_weather_final.to_csv(
    weather_output,
    index=False
)
print("Saved to:", weather_output)
print("File exists:", weather_output.exists())

Saved to: d:\Dengue_Project\data\interim\weather_monthly_prepared_2010_2024.csv
File exists: True


In [107]:
weather_check = pd.read_csv(
    weather_output
)
print(
    "Reloaded shape:",
    weather_check.shape
)
print(
    "Reloaded districts:",
    weather_check[
        "administrative_district"
    ].nunique()
)
print(
    "Missing values:",
    weather_check.isna().sum().sum()
)

Reloaded shape: (4500, 16)
Reloaded districts: 25
Missing values: 0


## Key Findings

1. The standardized weather dataset contains 20,332 weekly observations covering 26 reporting areas.

2. No standard missing values, common sentinel values, duplicate weekly geographic keys, invalid dates, chronological gaps or major physical consistency errors were identified.

3. The weekly weather sequence contains 782 consecutive reporting periods, with consecutive periods separated by exactly seven days.

4. A total of 4,030 weather observations (19.82%) cross calendar-month boundaries, demonstrating the need for explicit temporal aggregation rules.

5. Weather variables were aggregated according to their statistical meaning. Mean-state variables were converted using day-weighted means, accumulated quantities were proportionally allocated before monthly summation, and extrema were derived from overlapping weekly extrema.

6. The monthly reporting area dataset contains 4,680 observations, representing 26 reporting areas across 180 calendar months.

7. Temporal allocation preserves the total weekly rainfall and rainy-day quantities.

8. Ampara and Kalmunai are separate weather reporting locations corresponding to the same administrative district. Because weather measurements are not additive, their monthly weather summaries were combined using an unweighted arithmetic mean rather than summation.

9. The final administrative district weather dataset contains 4,500 observations representing 25 districts across 180 months.

10. No missing weather feature values or duplicate district month keys remain after preparation.

11. January 2010 and December 2024 have incomplete calendar-day coverage because the available weekly weather sequence begins on 2 January 2010 and ends on 27 December 2024. These 50 district month observations are explicitly flagged rather than silently removed.

12. The prepared monthly weather dataset is suitable for subsequent population preparation, missing value analysis and dataset integration, subject to later treatment of the flagged boundary months.

## Conclusion

The weekly weather dataset was successfully validated and transformed into monthly environmental features for dengue forecasting.

Temporal aggregation was performed according to the statistical meaning of each weather variable. Weekly mean temperature, humidity and wind speed were converted to day-weighted monthly means. Weekly rainfall totals and rainy-day counts were proportionally allocated across calendar months before summation. Weekly minimum and maximum variables were used to derive monthly extrema from reporting periods overlapping each month.

Because the source data contain weekly summaries rather than daily meteorological observations, temporal allocation across month boundaries requires assumptions. In particular, proportional allocation of accumulated variables assumes an even distribution within a reporting week, while the exact timing of weekly extrema is unknown. These limitations are retained explicitly in the methodology.

The 26 reporting-area weather series were subsequently converted to the 25 administrative districts required for integration with dengue and population data. Ampara and Kalmunai weather summaries were represented by their unweighted arithmetic mean because weather measurements are not additive.

The resulting dataset contains 4,500 district-month observations covering January 2010 through December 2024, with no missing weather feature values or duplicate district-month keys. January 2010 and December 2024 are explicitly flagged as incomplete weather months because the source reporting sequence does not cover every calendar day in those boundary months.

No raw weather files were modified during this preparation process.